In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\stuff\DATA MINING PROJECT")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print("ipl_etl exists:", (PROJECT_ROOT / "src" / "transformation" / "ipl_etl.py").exists())
print("Python can see project root:", str(PROJECT_ROOT) in sys.path)

Project root: C:\stuff\DATA MINING PROJECT
src exists: True
ipl_etl exists: False
Python can see project root: True


In [4]:
import json
import os
import pandas as pd
from pathlib import Path

In [10]:
IPL_PATH = Path(r"C:\stuff\DATA MINING PROJECT\data\raw\ipl_male_json")

print("Path exists:", IPL_PATH.exists())
print("Number of JSON files:", len(list(IPL_PATH.glob("*.json"))))

Path exists: True
Number of JSON files: 1243


In [11]:
json_files = sorted(IPL_PATH.glob("*.json"))

if json_files:
    first_match_file = json_files[0]
else:
    first_match_file = None
    print(f"No JSON files found in {IPL_PATH}")

print(first_match_file)

C:\stuff\DATA MINING PROJECT\data\raw\ipl_male_json\1082591.json


In [14]:
with open(first_match_file, "r", encoding="utf-8") as f:
    match = json.load(f)

print(match.keys())

dict_keys(['meta', 'info', 'innings'])


In [15]:
info = match["info"]

print("Season:", info["season"])
print("Date:", info["dates"])
print("Teams:", info["teams"])
print("Venue:", info["venue"])
print("City:", info.get("city"))

Season: 2017
Date: ['2017-04-05']
Teams: ['Sunrisers Hyderabad', 'Royal Challengers Bangalore']
Venue: Rajiv Gandhi International Stadium, Uppal
City: Hyderabad


In [16]:
print("Number of innings:", len(match["innings"]))

for i, innings in enumerate(match["innings"], start=1):
    print(f"Innings {i}: {innings['team']}")

Number of innings: 2
Innings 1: Sunrisers Hyderabad
Innings 2: Royal Challengers Bangalore


In [17]:
first_innings = match["innings"][0]
first_over = first_innings["overs"][0]
first_delivery = first_over["deliveries"][0]

print(first_delivery)

{'actual_delivery': '0.1', 'batter': 'DA Warner', 'bowler': 'TS Mills', 'non_striker': 'S Dhawan', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}


In [18]:
deliveries = []

for innings in match["innings"]:
    
    batting_team = innings["team"]
    
    for over in innings["overs"]:
        
        for delivery in over["deliveries"]:
            
            row = {
                "match_id": first_match_file.stem,
                "season": info["season"],
                "date": info["dates"][0],
                "venue": info["venue"],
                "city": info.get("city"),
                "batting_team": batting_team,
                "batter": delivery["batter"],
                "bowler": delivery["bowler"],
                "non_striker": delivery["non_striker"],
                "delivery": delivery["actual_delivery"],
                "batter_runs": delivery["runs"]["batter"],
                "extra_runs": delivery["runs"]["extras"],
                "total_runs": delivery["runs"]["total"]
            }
            
            deliveries.append(row)

deliveries_df = pd.DataFrame(deliveries)

print("Total deliveries:", len(deliveries_df))

Total deliveries: 248


In [19]:
deliveries_df.head(10)

,match_id,season,date,venue,city,batting_team,batter,bowler,non_striker,delivery,batter_runs,extra_runs,total_runs
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0.1,0,0,0
1,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0.2,0,0,0
2,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0.3,4,0,4
3,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0.4,0,0,0
4,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0.5,0,2,2
5,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,S Dhawan,TS Mills,DA Warner,0.5,0,0,0
6,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,S Dhawan,TS Mills,DA Warner,0.6,0,1,1
7,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,S Dhawan,A Choudhary,DA Warner,1.1,1,0,1
8,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,A Choudhary,S Dhawan,1.2,4,0,4
9,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,A Choudhary,S Dhawan,1.3,0,1,1


In [20]:
def is_legal_ball(delivery):
    extras = delivery.get("extras", {})
    
    return (
        "wides" not in extras
        and "noballs" not in extras
    )

In [21]:
deliveries = []

for innings in match["innings"]:
    
    batting_team = innings["team"]
    
    for over in innings["overs"]:
        
        for delivery in over["deliveries"]:
            
            extras = delivery.get("extras", {})
            
            row = {
                "match_id": first_match_file.stem,
                "season": info["season"],
                "date": info["dates"][0],
                "venue": info["venue"],
                "city": info.get("city"),
                "batting_team": batting_team,
                "batter": delivery["batter"],
                "bowler": delivery["bowler"],
                "non_striker": delivery["non_striker"],
                "delivery": delivery["actual_delivery"],
                "batter_runs": delivery["runs"]["batter"],
                "extra_runs": delivery["runs"]["extras"],
                "total_runs": delivery["runs"]["total"],
                "is_legal_ball": is_legal_ball(delivery),
                "extras": extras
            }
            
            deliveries.append(row)

deliveries_df = pd.DataFrame(deliveries)

deliveries_df.head()

,match_id,season,date,venue,city,batting_team,batter,bowler,non_striker,delivery,batter_runs,extra_runs,total_runs,is_legal_ball,extras
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0.1,0,0,0,True,{}
1,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0.2,0,0,0,True,{}
2,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0.3,4,0,4,True,{}
3,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0.4,0,0,0,True,{}
4,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0.5,0,2,2,False,{'wides': 2}


In [22]:
deliveries_df["four"] = (
    deliveries_df["batter_runs"] == 4
).astype(int)

deliveries_df["six"] = (
    deliveries_df["batter_runs"] == 6
).astype(int)

In [23]:
batting_match = (
    deliveries_df
    .groupby(
        [
            "match_id",
            "season",
            "batting_team",
            "batter"
        ],
        as_index=False
    )
    .agg(
        runs=("batter_runs", "sum"),
        balls=("is_legal_ball", "sum"),
        fours=("four", "sum"),
        sixes=("six", "sum")
    )
)

In [24]:
batting_match["strike_rate"] = (
    batting_match["runs"] /
    batting_match["balls"] *
    100
).round(2)

In [25]:
batting_match

,match_id,season,batting_team,batter,runs,balls,fours,sixes,strike_rate
0,1082591,2017,Royal Challengers Bangalore,A Choudhary,6,2,0,1,300.00
1,1082591,2017,Royal Challengers Bangalore,CH Gayle,32,21,2,3,152.38
2,1082591,2017,Royal Challengers Bangalore,KM Jadhav,31,15,4,1,206.67
3,1082591,2017,Royal Challengers Bangalore,Mandeep Singh,24,16,5,0,150.00
4,1082591,2017,Royal Challengers Bangalore,S Aravind,0,2,0,0,0.00
5,1082591,2017,Royal Challengers Bangalore,SR Watson,22,17,1,1,129.41
6,1082591,2017,Royal Challengers Bangalore,STR Binny,11,10,0,1,110.00
7,1082591,2017,Royal Challengers Bangalore,Sachin Baby,1,3,0,0,33.33
8,1082591,2017,Royal Challengers Bangalore,TM Head,30,22,3,0,136.36
9,1082591,2017,Royal Challengers Bangalore,TS Mills,6,3,0,1,200.00


In [26]:
deliveries_df["extras"].head(20)

0                 {}
1                 {}
2                 {}
3                 {}
4       {'wides': 2}
5                 {}
6     {'legbyes': 1}
7                 {}
8                 {}
9     {'noballs': 1}
10                {}
11                {}
12                {}
13                {}
14                {}
15                {}
16                {}
17                {}
18                {}
19                {}
Name: extras, dtype: object

In [27]:
deliveries_df[
    deliveries_df["extras"].apply(lambda x: len(x) > 0)
][["batter", "bowler", "batter_runs", "extra_runs", "extras"]].head(20)

,batter,bowler,batter_runs,extra_runs,extras
4,DA Warner,TS Mills,0,2,{'wides': 2}
6,S Dhawan,TS Mills,0,1,{'legbyes': 1}
9,DA Warner,A Choudhary,0,1,{'noballs': 1}
78,Yuvraj Singh,A Choudhary,0,1,{'wides': 1}
107,Yuvraj Singh,A Choudhary,0,1,{'wides': 1}
108,Yuvraj Singh,A Choudhary,0,1,{'wides': 1}
146,CH Gayle,B Kumar,0,1,{'wides': 1}
153,CH Gayle,BCJ Cutting,0,1,{'wides': 1}
176,KM Jadhav,BCJ Cutting,0,1,{'wides': 1}
182,KM Jadhav,MC Henriques,1,1,{'noballs': 1}


In [31]:
def calculate_bowler_runs(row):
    total = row["total_runs"]
    extras = row["extras"]

    # Byes and leg-byes are not charged to the bowler
    total -= extras.get("byes", 0)
    total -= extras.get("legbyes", 0)
    
    # Penalty runs are not charged to the bowler
    total -= extras.get("penalty", 0)

    return total

In [32]:
deliveries_df["bowler_runs"] = deliveries_df.apply(
    calculate_bowler_runs,
    axis=1
)


In [33]:
deliveries_df[
    ["batter", "bowler", "total_runs", "extras", "bowler_runs"]
].head(20)

,batter,bowler,total_runs,extras,bowler_runs
0,DA Warner,TS Mills,0,{},0
1,DA Warner,TS Mills,0,{},0
2,DA Warner,TS Mills,4,{},4
3,DA Warner,TS Mills,0,{},0
4,DA Warner,TS Mills,2,{'wides': 2},2
5,S Dhawan,TS Mills,0,{},0
6,S Dhawan,TS Mills,1,{'legbyes': 1},0
7,S Dhawan,A Choudhary,1,{},1
8,DA Warner,A Choudhary,4,{},4
9,DA Warner,A Choudhary,1,{'noballs': 1},1


In [34]:
bowling_match = (
    deliveries_df
    .groupby(
        [
            "match_id",
            "season",
            "batting_team",
            "bowler"
        ],
        as_index=False
    )
    .agg(
        balls=("is_legal_ball", "sum"),
        runs_conceded=("bowler_runs", "sum")
    )
)

bowling_match

,match_id,season,batting_team,bowler,balls,runs_conceded
0,1082591,2017,Royal Challengers Bangalore,A Nehra,24,42
1,1082591,2017,Royal Challengers Bangalore,B Kumar,24,27
2,1082591,2017,Royal Challengers Bangalore,BCJ Cutting,22,35
3,1082591,2017,Royal Challengers Bangalore,Bipul Sharma,6,4
4,1082591,2017,Royal Challengers Bangalore,DJ Hooda,6,7
5,1082591,2017,Royal Challengers Bangalore,MC Henriques,12,20
6,1082591,2017,Royal Challengers Bangalore,Rashid Khan,24,36
7,1082591,2017,Sunrisers Hyderabad,A Choudhary,24,55
8,1082591,2017,Sunrisers Hyderabad,S Aravind,18,36
9,1082591,2017,Sunrisers Hyderabad,SR Watson,18,41


In [35]:
bowling_match["overs"] = (
    (bowling_match["balls"] // 6).astype(str)
    + "."
    + (bowling_match["balls"] % 6).astype(str)
)

bowling_match["economy"] = (
    bowling_match["runs_conceded"]
    / bowling_match["balls"]
    * 6
).round(2)

bowling_match

,match_id,season,batting_team,bowler,balls,runs_conceded,overs,economy
0,1082591,2017,Royal Challengers Bangalore,A Nehra,24,42,4.0,10.50
1,1082591,2017,Royal Challengers Bangalore,B Kumar,24,27,4.0,6.75
2,1082591,2017,Royal Challengers Bangalore,BCJ Cutting,22,35,3.4,9.55
3,1082591,2017,Royal Challengers Bangalore,Bipul Sharma,6,4,1.0,4.00
4,1082591,2017,Royal Challengers Bangalore,DJ Hooda,6,7,1.0,7.00
5,1082591,2017,Royal Challengers Bangalore,MC Henriques,12,20,2.0,10.00
6,1082591,2017,Royal Challengers Bangalore,Rashid Khan,24,36,4.0,9.00
7,1082591,2017,Sunrisers Hyderabad,A Choudhary,24,55,4.0,13.75
8,1082591,2017,Sunrisers Hyderabad,S Aravind,18,36,3.0,12.00
9,1082591,2017,Sunrisers Hyderabad,SR Watson,18,41,3.0,13.67


In [36]:
for innings in match["innings"]:
    for over in innings["overs"]:
        for delivery in over["deliveries"]:
            
            if "wickets" in delivery:
                print(
                    "Bowler:", delivery["bowler"],
                    "| Wickets:", delivery["wickets"]
                )

Bowler: A Choudhary | Wickets: [{'kind': 'caught', 'player_out': 'DA Warner', 'fielders': [{'name': 'Mandeep Singh'}]}]
Bowler: STR Binny | Wickets: [{'kind': 'caught', 'player_out': 'S Dhawan', 'fielders': [{'name': 'Sachin Baby'}]}]
Bowler: YS Chahal | Wickets: [{'kind': 'caught', 'player_out': 'MC Henriques', 'fielders': [{'name': 'Sachin Baby'}]}]
Bowler: TS Mills | Wickets: [{'kind': 'bowled', 'player_out': 'Yuvraj Singh'}]
Bowler: Rashid Khan | Wickets: [{'kind': 'bowled', 'player_out': 'Mandeep Singh'}]
Bowler: DJ Hooda | Wickets: [{'kind': 'caught', 'player_out': 'CH Gayle', 'fielders': [{'name': 'DA Warner'}]}]
Bowler: MC Henriques | Wickets: [{'kind': 'run out', 'player_out': 'KM Jadhav', 'fielders': [{'name': 'BCJ Cutting'}]}]
Bowler: Rashid Khan | Wickets: [{'kind': 'caught', 'player_out': 'TM Head', 'fielders': [{'name': 'Yuvraj Singh'}]}]
Bowler: Bipul Sharma | Wickets: [{'kind': 'caught', 'player_out': 'Sachin Baby', 'fielders': [{'name': 'MC Henriques'}]}]
Bowler: B Kum

In [37]:
BOWLER_WICKET_TYPES = {
    "bowled",
    "caught",
    "caught and bowled",
    "lbw",
    "stumped",
    "hit wicket"
}

In [38]:
def get_bowler_wickets(delivery):
    wickets = delivery.get("wickets", [])
    
    count = 0
    
    for wicket in wickets:
        if wicket["kind"] in BOWLER_WICKET_TYPES:
            count += 1
    
    return count

In [39]:
deliveries = []

for innings in match["innings"]:
    
    batting_team = innings["team"]
    
    for over in innings["overs"]:
        
        for delivery in over["deliveries"]:
            
            extras = delivery.get("extras", {})
            wickets = delivery.get("wickets", [])
            
            row = {
                "match_id": first_match_file.stem,
                "season": info["season"],
                "date": info["dates"][0],
                "venue": info["venue"],
                "city": info.get("city"),
                "batting_team": batting_team,
                "batter": delivery["batter"],
                "bowler": delivery["bowler"],
                "non_striker": delivery["non_striker"],
                "delivery": delivery["actual_delivery"],
                "batter_runs": delivery["runs"]["batter"],
                "extra_runs": delivery["runs"]["extras"],
                "total_runs": delivery["runs"]["total"],
                "is_legal_ball": is_legal_ball(delivery),
                "extras": extras,
                "wickets": wickets
            }
            
            deliveries.append(row)

deliveries_df = pd.DataFrame(deliveries)

print("Total deliveries:", len(deliveries_df))

Total deliveries: 248


In [40]:
deliveries_df["bowler_wickets"] = deliveries_df.apply(
    get_bowler_wickets,
    axis=1
)

In [41]:
deliveries_df[
    deliveries_df["bowler_wickets"] > 0
][
    ["bowler", "batter", "wickets", "bowler_wickets"]
]

,bowler,batter,wickets,bowler_wickets
11,A Choudhary,DA Warner,"[{'kind': 'caught', 'player_out': 'DA Warner',...",1
64,STR Binny,S Dhawan,"[{'kind': 'caught', 'player_out': 'S Dhawan', ...",1
94,YS Chahal,MC Henriques,"[{'kind': 'caught', 'player_out': 'MC Henrique...",1
116,TS Mills,Yuvraj Singh,"[{'kind': 'bowled', 'player_out': 'Yuvraj Sing...",1
160,Rashid Khan,Mandeep Singh,"[{'kind': 'bowled', 'player_out': 'Mandeep Sin...",1
165,DJ Hooda,CH Gayle,"[{'kind': 'caught', 'player_out': 'CH Gayle', ...",1
206,Rashid Khan,TM Head,"[{'kind': 'caught', 'player_out': 'TM Head', '...",1
211,Bipul Sharma,Sachin Baby,"[{'kind': 'caught', 'player_out': 'Sachin Baby...",1
230,B Kumar,STR Binny,"[{'kind': 'caught', 'player_out': 'STR Binny',...",1
234,A Nehra,SR Watson,"[{'kind': 'caught', 'player_out': 'SR Watson',...",1


In [43]:
# Ensure the required columns exist before grouping
if "bowler_runs" not in deliveries_df.columns:
    def calculate_bowler_runs(row):
        total = row["total_runs"]
        extras = row.get("extras", {})
        total -= extras.get("byes", 0)
        total -= extras.get("legbyes", 0)
        total -= extras.get("penalty", 0)
        return total

    deliveries_df["bowler_runs"] = deliveries_df.apply(calculate_bowler_runs, axis=1)

if "bowler_wickets" not in deliveries_df.columns:
    def get_bowler_wickets(delivery):
        wickets = delivery.get("wickets", [])
        count = 0
        for wicket in wickets:
            if wicket["kind"] in BOWLER_WICKET_TYPES:
                count += 1
        return count

    deliveries_df["bowler_wickets"] = deliveries_df.apply(get_bowler_wickets, axis=1)

bowling_match = (
    deliveries_df
    .groupby(
        ["match_id", "season", "batting_team", "bowler"],
        as_index=False
    )
    .agg(
        balls=("is_legal_ball", "sum"),
        runs_conceded=("bowler_runs", "sum"),
        wickets=("bowler_wickets", "sum")
    )
)

bowling_match["overs"] = (
    (bowling_match["balls"] // 6).astype(str)
    + "."
    + (bowling_match["balls"] % 6).astype(str)
)

bowling_match["economy"] = (
    bowling_match["runs_conceded"] / bowling_match["balls"] * 6
).round(2)

bowling_match

,match_id,season,batting_team,bowler,balls,runs_conceded,wickets,overs,economy
0,1082591,2017,Royal Challengers Bangalore,A Nehra,24,42,2,4.0,10.50
1,1082591,2017,Royal Challengers Bangalore,B Kumar,24,27,2,4.0,6.75
2,1082591,2017,Royal Challengers Bangalore,BCJ Cutting,22,35,0,3.4,9.55
3,1082591,2017,Royal Challengers Bangalore,Bipul Sharma,6,4,1,1.0,4.00
4,1082591,2017,Royal Challengers Bangalore,DJ Hooda,6,7,1,1.0,7.00
5,1082591,2017,Royal Challengers Bangalore,MC Henriques,12,20,0,2.0,10.00
6,1082591,2017,Royal Challengers Bangalore,Rashid Khan,24,36,2,4.0,9.00
7,1082591,2017,Sunrisers Hyderabad,A Choudhary,24,55,1,4.0,13.75
8,1082591,2017,Sunrisers Hyderabad,S Aravind,18,36,0,3.0,12.00
9,1082591,2017,Sunrisers Hyderabad,SR Watson,18,41,0,3.0,13.67


In [44]:
bowling_match["overs"] = (
    (bowling_match["balls"] // 6).astype(str)
    + "."
    + (bowling_match["balls"] % 6).astype(str)
)

bowling_match["economy"] = (
    bowling_match["runs_conceded"]
    / bowling_match["balls"]
    * 6
).round(2)

In [45]:
bowling_match[
    [
        "bowler",
        "balls",
        "overs",
        "runs_conceded",
        "wickets",
        "economy"
    ]
]

,bowler,balls,overs,runs_conceded,wickets,economy
0,A Nehra,24,4.0,42,2,10.50
1,B Kumar,24,4.0,27,2,6.75
2,BCJ Cutting,22,3.4,35,0,9.55
3,Bipul Sharma,6,1.0,4,1,4.00
4,DJ Hooda,6,1.0,7,1,7.00
5,MC Henriques,12,2.0,20,0,10.00
6,Rashid Khan,24,4.0,36,2,9.00
7,A Choudhary,24,4.0,55,1,13.75
8,S Aravind,18,3.0,36,0,12.00
9,SR Watson,18,3.0,41,0,13.67


In [47]:
bowling_match["strike_rate"] = (
    bowling_match["balls"]
    .astype("float64")
    .div(bowling_match["wickets"].astype("float64"))
    .where(bowling_match["wickets"].ne(0))
    .round(2)
)

In [48]:
bowling_match[
    [
        "bowler",
        "overs",
        "runs_conceded",
        "wickets",
        "economy"
    ]
].sort_values(
    "wickets",
    ascending=False
)

,bowler,overs,runs_conceded,wickets,economy
0,A Nehra,4.0,42,2,10.50
1,B Kumar,4.0,27,2,6.75
6,Rashid Khan,4.0,36,2,9.00
3,Bipul Sharma,1.0,4,1,4.00
7,A Choudhary,4.0,55,1,13.75
4,DJ Hooda,1.0,7,1,7.00
13,YS Chahal,4.0,22,1,5.50
10,STR Binny,1.0,10,1,10.00
12,TS Mills,4.0,31,1,7.75
2,BCJ Cutting,3.4,35,0,9.55


In [49]:
BOWLER_WICKET_TYPES = {
    "bowled",
    "caught",
    "caught and bowled",
    "lbw",
    "stumped",
    "hit wicket"
}


def is_legal_ball(delivery):
    extras = delivery.get("extras", {})
    
    return (
        "wides" not in extras
        and "noballs" not in extras
    )


def calculate_bowler_runs(delivery):
    total = delivery["runs"]["total"]
    extras = delivery.get("extras", {})

    total -= extras.get("byes", 0)
    total -= extras.get("legbyes", 0)
    total -= extras.get("penalty", 0)

    return total


def get_bowler_wickets(delivery):
    wickets = delivery.get("wickets", [])

    return sum(
        1
        for wicket in wickets
        if wicket["kind"] in BOWLER_WICKET_TYPES
    )

In [53]:
def process_ipl_match(file_path):

    # -------------------------
    # Load match
    # -------------------------

    with open(file_path, "r", encoding="utf-8") as f:
        match = json.load(f)

    info = match["info"]

    match_id = file_path.stem
    season = info["season"]
    date = info["dates"][0]
    teams = info["teams"]
    venue = info["venue"]
    city = info.get("city")


    # -------------------------
    # Extract deliveries
    # -------------------------

    deliveries = []

    for innings in match["innings"]:

        batting_team = innings["team"]

        for over in innings["overs"]:

            for delivery in over["deliveries"]:

                extras = delivery.get("extras", {})

                deliveries.append({
                    "match_id": match_id,
                    "season": season,
                    "date": date,
                    "venue": venue,
                    "city": city,
                    "batting_team": batting_team,
                    "batter": delivery["batter"],
                    "bowler": delivery["bowler"],
                    "non_striker": delivery["non_striker"],
                    "delivery": delivery["actual_delivery"],
                    "batter_runs": delivery["runs"]["batter"],
                    "extra_runs": delivery["runs"]["extras"],
                    "total_runs": delivery["runs"]["total"],
                    "is_legal_ball": is_legal_ball(delivery),
                    "bowler_runs": calculate_bowler_runs(delivery),
                    "bowler_wickets": get_bowler_wickets(delivery),
                    "four": int(delivery["runs"]["batter"] == 4),
                    "six": int(delivery["runs"]["batter"] == 6)
                })


    deliveries_df = pd.DataFrame(deliveries)


    # -------------------------
    # Batting statistics
    # -------------------------

    batting_match = (
        deliveries_df
        .groupby(
            [
                "match_id",
                "season",
                "date",
                "batting_team",
                "batter"
            ],
            as_index=False
        )
        .agg(
            runs=("batter_runs", "sum"),
            balls=("is_legal_ball", "sum"),
            fours=("four", "sum"),
            sixes=("six", "sum")
        )
    )

    # Avoid division by zero
    batting_match["strike_rate"] = (
        batting_match["runs"]
        .div(batting_match["balls"].replace(0, float("nan")))
        .mul(100)
        .round(2)
    )


    # -------------------------
    # Bowling statistics
    # -------------------------

    bowling_match = (
        deliveries_df
        .groupby(
            [
                "match_id",
                "season",
                "date",
                "batting_team",
                "bowler"
            ],
            as_index=False
        )
        .agg(
            balls=("is_legal_ball", "sum"),
            runs_conceded=("bowler_runs", "sum"),
            wickets=("bowler_wickets", "sum")
        )
    )


    # -------------------------
    # Overs
    # -------------------------

    bowling_match["overs"] = (
        (bowling_match["balls"] // 6).astype(str)
        + "."
        + (bowling_match["balls"] % 6).astype(str)
    )


    # -------------------------
    # Economy
    # -------------------------

    bowling_match["economy"] = (
        bowling_match["runs_conceded"]
        .div(bowling_match["balls"].replace(0, float("nan")))
        .mul(6)
        .round(2)
    )


    # -------------------------
    # Bowling strike rate
    # -------------------------

    bowling_match["bowling_strike_rate"] = (
        bowling_match["balls"]
        .div(
            bowling_match["wickets"].replace(
                0,
                float("nan")
            )
        )
        .round(2)
    )


    # -------------------------
    # Match information
    # -------------------------

    match_data = pd.DataFrame([{
        "match_id": match_id,
        "season": season,
        "date": date,
        "team1": teams[0],
        "team2": teams[1],
        "venue": venue,
        "city": city
    }])


    # -------------------------
    # Return results
    # -------------------------

    return match_data, batting_match, bowling_match

In [54]:
match_test, batting_test, bowling_test = process_ipl_match(
    first_match_file
)
print("Match:")
display(match_test)

print("Batting:")
display(batting_test)

print("Bowling:")
display(bowling_test)

Match:


,match_id,season,date,team1,team2,venue,city
0,1082591,2017,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",Hyderabad


Batting:


,match_id,season,date,batting_team,batter,runs,balls,fours,sixes,strike_rate
0,1082591,2017,2017-04-05,Royal Challengers Bangalore,A Choudhary,6,2,0,1,300.00
1,1082591,2017,2017-04-05,Royal Challengers Bangalore,CH Gayle,32,21,2,3,152.38
2,1082591,2017,2017-04-05,Royal Challengers Bangalore,KM Jadhav,31,15,4,1,206.67
3,1082591,2017,2017-04-05,Royal Challengers Bangalore,Mandeep Singh,24,16,5,0,150.00
4,1082591,2017,2017-04-05,Royal Challengers Bangalore,S Aravind,0,2,0,0,0.00
5,1082591,2017,2017-04-05,Royal Challengers Bangalore,SR Watson,22,17,1,1,129.41
6,1082591,2017,2017-04-05,Royal Challengers Bangalore,STR Binny,11,10,0,1,110.00
7,1082591,2017,2017-04-05,Royal Challengers Bangalore,Sachin Baby,1,3,0,0,33.33
8,1082591,2017,2017-04-05,Royal Challengers Bangalore,TM Head,30,22,3,0,136.36
9,1082591,2017,2017-04-05,Royal Challengers Bangalore,TS Mills,6,3,0,1,200.00


Bowling:


,match_id,season,date,batting_team,bowler,balls,runs_conceded,wickets,overs,economy,bowling_strike_rate
0,1082591,2017,2017-04-05,Royal Challengers Bangalore,A Nehra,24,42,2,4.0,10.50,12.0
1,1082591,2017,2017-04-05,Royal Challengers Bangalore,B Kumar,24,27,2,4.0,6.75,12.0
2,1082591,2017,2017-04-05,Royal Challengers Bangalore,BCJ Cutting,22,35,0,3.4,9.55,NaN
3,1082591,2017,2017-04-05,Royal Challengers Bangalore,Bipul Sharma,6,4,1,1.0,4.00,6.0
4,1082591,2017,2017-04-05,Royal Challengers Bangalore,DJ Hooda,6,7,1,1.0,7.00,6.0
5,1082591,2017,2017-04-05,Royal Challengers Bangalore,MC Henriques,12,20,0,2.0,10.00,NaN
6,1082591,2017,2017-04-05,Royal Challengers Bangalore,Rashid Khan,24,36,2,4.0,9.00,12.0
7,1082591,2017,2017-04-05,Sunrisers Hyderabad,A Choudhary,24,55,1,4.0,13.75,24.0
8,1082591,2017,2017-04-05,Sunrisers Hyderabad,S Aravind,18,36,0,3.0,12.00,NaN
9,1082591,2017,2017-04-05,Sunrisers Hyderabad,SR Watson,18,41,0,3.0,13.67,NaN


In [55]:
match_test, batting_test, bowling_test = process_ipl_match(
    first_match_file
)

In [56]:
print("MATCH")
display(match_test)

print("BATTING")
display(batting_test)

print("BOWLING")
display(bowling_test)

MATCH


,match_id,season,date,team1,team2,venue,city
0,1082591,2017,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",Hyderabad


BATTING


,match_id,season,date,batting_team,batter,runs,balls,fours,sixes,strike_rate
0,1082591,2017,2017-04-05,Royal Challengers Bangalore,A Choudhary,6,2,0,1,300.00
1,1082591,2017,2017-04-05,Royal Challengers Bangalore,CH Gayle,32,21,2,3,152.38
2,1082591,2017,2017-04-05,Royal Challengers Bangalore,KM Jadhav,31,15,4,1,206.67
3,1082591,2017,2017-04-05,Royal Challengers Bangalore,Mandeep Singh,24,16,5,0,150.00
4,1082591,2017,2017-04-05,Royal Challengers Bangalore,S Aravind,0,2,0,0,0.00
5,1082591,2017,2017-04-05,Royal Challengers Bangalore,SR Watson,22,17,1,1,129.41
6,1082591,2017,2017-04-05,Royal Challengers Bangalore,STR Binny,11,10,0,1,110.00
7,1082591,2017,2017-04-05,Royal Challengers Bangalore,Sachin Baby,1,3,0,0,33.33
8,1082591,2017,2017-04-05,Royal Challengers Bangalore,TM Head,30,22,3,0,136.36
9,1082591,2017,2017-04-05,Royal Challengers Bangalore,TS Mills,6,3,0,1,200.00


BOWLING


,match_id,season,date,batting_team,bowler,balls,runs_conceded,wickets,overs,economy,bowling_strike_rate
0,1082591,2017,2017-04-05,Royal Challengers Bangalore,A Nehra,24,42,2,4.0,10.50,12.0
1,1082591,2017,2017-04-05,Royal Challengers Bangalore,B Kumar,24,27,2,4.0,6.75,12.0
2,1082591,2017,2017-04-05,Royal Challengers Bangalore,BCJ Cutting,22,35,0,3.4,9.55,NaN
3,1082591,2017,2017-04-05,Royal Challengers Bangalore,Bipul Sharma,6,4,1,1.0,4.00,6.0
4,1082591,2017,2017-04-05,Royal Challengers Bangalore,DJ Hooda,6,7,1,1.0,7.00,6.0
5,1082591,2017,2017-04-05,Royal Challengers Bangalore,MC Henriques,12,20,0,2.0,10.00,NaN
6,1082591,2017,2017-04-05,Royal Challengers Bangalore,Rashid Khan,24,36,2,4.0,9.00,12.0
7,1082591,2017,2017-04-05,Sunrisers Hyderabad,A Choudhary,24,55,1,4.0,13.75,24.0
8,1082591,2017,2017-04-05,Sunrisers Hyderabad,S Aravind,18,36,0,3.0,12.00,NaN
9,1082591,2017,2017-04-05,Sunrisers Hyderabad,SR Watson,18,41,0,3.0,13.67,NaN


In [7]:
import sys
from pathlib import Path
import importlib.util

PROJECT_ROOT = Path(r"C:\stuff\DATA MINING PROJECT")

ETL_FILE = PROJECT_ROOT / "src" / "transformation" / "etl1.py"

print("ETL file exists:", ETL_FILE.exists())
print("ETL file:", ETL_FILE)

ETL file exists: False
ETL file: C:\stuff\DATA MINING PROJECT\src\transformation\etl1.py


In [9]:
spec = importlib.util.spec_from_file_location(
    "etl1",
    ETL_FILE
)

etl1 = importlib.util.module_from_spec(spec)

spec.loader.exec_module(etl1)

print("ETL module loaded successfully!")

ETL module loaded successfully!


In [10]:
print(hasattr(etl1, "process_ipl_match"))
print(hasattr(etl1, "process_all_ipl_matches"))

True
True


In [11]:
IPL_RAW = Path(
    r"C:\stuff\DATA MINING PROJECT\data\raw\ipl_male_json"
)

IPL_PROCESSED = Path(
    r"C:\stuff\DATA MINING PROJECT\data\processed\ipl"
)

matches_df, batting_df, bowling_df = etl1.process_all_ipl_matches(
    IPL_RAW,
    IPL_PROCESSED
)

Found 1243 IPL match files.
Processed 100/1243 matches...
Processed 200/1243 matches...
Processed 300/1243 matches...
Processed 400/1243 matches...
Processed 500/1243 matches...
Processed 600/1243 matches...
Processed 700/1243 matches...
Processed 800/1243 matches...
Processed 900/1243 matches...
Processed 1000/1243 matches...
Processed 1100/1243 matches...
Processed 1200/1243 matches...

ETL COMPLETE
-------------------------
Matches processed: 1243
Batting records: 18768
Bowling records: 14700
Unique players: 806
Errors: 0


In [14]:
print("Matches:", len(matches_df))
print("Batting records:", len(batting_df))
print("Bowling records:", len(bowling_df))

print("Unique batting players:", batting_df["player"].nunique())
print("Unique bowling players:", bowling_df["player"].nunique())

print("\nIPL seasons:")
print(print(
    sorted(
        matches_df["season"].astype(str).unique()
    )
))

Matches: 1243
Batting records: 18768
Bowling records: 14700
Unique batting players: 738
Unique bowling players: 577

IPL seasons:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']
None


In [15]:
print("Duplicate match records:",
      matches_df["match_id"].duplicated().sum())

print("Duplicate batting records:",
      batting_df.duplicated().sum())

print("Duplicate bowling records:",
      bowling_df.duplicated().sum())

Duplicate match records: 0
Duplicate batting records: 0
Duplicate bowling records: 0


In [16]:
error_file = IPL_PROCESSED / "etl_errors.csv"

print("Error file exists:", error_file.exists())

if error_file.exists():
    errors_df = pd.read_csv(error_file)
    print("Number of ETL errors:", len(errors_df))
    display(errors_df.head(20))

Error file exists: False
